# Practice Resolution - Business activity trends

Using the Business Activity Trends public dataset availale at [HDX](https://data.humdata.org/dataset/facebook-business-activity-trends-during-covid19?fbclid=IwZXh0bgNhZW0CMTEAAR1FMJeXzPcttgPISP6SQp0LIuBCiKlc8Qvl75LfOYmn16QEpMPLlSUz0fw_aem_axfaii-50SG3O4PrMDF5lA) plot the activity quantile for 3 countries:
- Make a single plot with the vertical `all` for the three countries.
- Select 3 activities of you interest and create a figure with 3 subplots, one per activity, showing the results for the three already selected countries.

Can you extract any conclusions from the plot? 

Note: This dataset is only available at the national level

In [15]:
import os
import pandas as pd
from datetime import datetime

from bokeh.plotting import figure, output_file, show
from bokeh.models import Legend, ColumnDataSource, Span, Tabs, TabPanel
from bokeh.layouts import column, gridplot
from bokeh.io import output_notebook
from bokeh.core.validation.warnings import MISSING_RENDERERS, EMPTY_LAYOUT

import bokeh

In [23]:
data_dir = "C:/Users/Usuario/Downloads/business-activity-trends-during-covid-19-20200301-20221129"

selected_countries = ["Argentina", "Ethiopia", "Nepal"] 

# Load and merge csv files
all_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".csv")]
df = [pd.read_csv(file) for file in all_files]
data = pd.concat(df, ignore_index=True)

In [24]:
# Filter for selected_countries
data = data[data["gadm0_name"].isin(selected_countries)]
data["ds"] = pd.to_datetime(data["ds"]) 

In [25]:
data

,gadm_id,gadm_name,gadm_level,gadm0_name,gadm1_name,gadm2_name,country,business_vertical,activity_quantile,activity_percentage,crisis_ds,ds
127,NPL,Nepal,0,Nepal,NaN,NaN,NP,Local Events,0.257274,95.638730,2020-03-01,2020-03-01
128,NPL,Nepal,0,Nepal,NaN,NaN,NP,Public Good,0.518314,105.137179,2020-03-01,2020-03-01
129,NPL,Nepal,0,Nepal,NaN,NaN,NP,Lifestyle Services,0.746596,121.582142,2020-03-01,2020-03-01
130,NPL,Nepal,0,Nepal,NaN,NaN,NP,Business & Utility Services,0.418353,105.310039,2020-03-01,2020-03-01
131,NPL,Nepal,0,Nepal,NaN,NaN,NP,Home Services,0.474936,110.704084,2020-03-01,2020-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...
2396125,ARG,Argentina,0,Argentina,NaN,NaN,AR,Lifestyle Services,0.128511,80.439195,2020-03-01,2022-11-29
2396126,ARG,Argentina,0,Argentina,NaN,NaN,AR,Professional Services,0.402973,110.858965,2020-03-01,2022-11-29
2396127,ARG,Argentina,0,Argentina,NaN,NaN,AR,All,0.372607,115.806496,2020-03-01,2022-11-29
2396128,ARG,Argentina,0,Argentina,NaN,NaN,AR,Home Services,0.323748,116.922346,2020-03-01,2022-11-29


In [4]:
duplicates = data.duplicated(subset=["ds", "gadm0_name", "business_vertical"], keep=False) # Check duplicates
data[duplicates]

,gadm_id,gadm_name,gadm_level,gadm0_name,gadm1_name,gadm2_name,country,business_vertical,activity_quantile,activity_percentage,crisis_ds,ds


## Plot with the vertical 'All'

In [31]:
color_palette = [
    "#4E79A7",  # Blue
    "#F28E2B",  # Orange
    "#E15759",  # Red
    "#76B7B2",  # Teal
    "#59A14F",  # Green
    "#EDC948",  # Yellow
    "#B07AA1",  # Purple
    "#FF9DA7",  # Pink
    "#9C755F",  # Brown
    "#BAB0AC",  # Gray
    "#7C7C7C",  # Dark gray
    "#6B4C9A",  # Violet
    "#D55E00",  # Orange-red
    "#CC61B0",  # Magenta
    "#0072B2",  # Bright blue
    "#329262",  # Peacock green
    "#9E5B5A",  # Brick red
    "#636363",  # Medium gray
    "#CD9C00",  # Gold
    "#5D69B1",  # Medium blue
]

def get_line_plot(businessActivity, title, source, earthquakes=False, subtitle=None):
    p2 = figure(x_axis_type="datetime", width=1500, height=600, toolbar_location="above")
    p2.add_layout(Legend(), "right")

    for id, country in enumerate(businessActivity["gadm0_name"].unique()):
        df = businessActivity[
            (businessActivity["gadm0_name"] == country)
            & (businessActivity["business_vertical"] == "All")
        ][["ds", "activity_quantile"]]

        p2.line(
            x=df["ds"],
            y=df["activity_quantile"],
            line_width=2,
            line_color=color_palette[id % len(color_palette)],
            legend_label=country,
        )

    p2.legend.click_policy = "hide"
    if subtitle is not None:
        p2.title = subtitle

    title_fig = figure(
        title=title,
        toolbar_location=None,
        width=800,
        height=40,
    )
    title_fig.title.align = "left"
    title_fig.title.text_font_size = "20pt"
    title_fig.border_fill_alpha = 0
    title_fig.outline_line_width = 0

    sub_title = figure(
        title=source,
        toolbar_location=None,
        width=800,
        height=40,
    )
    sub_title.title.align = "left"
    sub_title.title.text_font_size = "10pt"
    sub_title.title.text_font_style = "normal"
    sub_title.border_fill_alpha = 0
    sub_title.outline_line_width = 0

    if earthquakes:
        p2.renderers.extend(
            [
                Span(
                    location=datetime(2023, 2, 6),
                    dimension="height",
                    line_color="#7C7C7C",
                    line_width=2,
                    line_dash=(4, 4),
                ),
                Span(
                    location=datetime(2023, 2, 20),
                    dimension="height",
                    line_color="#7C7C7C",
                    line_width=2,
                    line_dash=(4, 4),
                ),
            ]
        )

    layout = column(title_fig, p2, sub_title)
    return layout

In [32]:
output_notebook()
bokeh.core.validation.silence(EMPTY_LAYOUT, True)
bokeh.core.validation.silence(MISSING_RENDERERS, True)

filtered_data = data[data["gadm0_name"].isin(selected_countries)]

tabs = []
tabs.append(
    TabPanel(
        child=get_line_plot(
            filtered_data,
            "Activity Quantile Comparison during Covid",
            "Source: Data for Good Meta",
            earthquakes=True,
            subtitle="Comparison of activity quantile for the 'All' vertical",
        ),
        #title="Comparison",
    )
)

tabs = Tabs(tabs=tabs, sizing_mode="scale_both")
show(tabs)

Loading BokehJS ...

## Plot with selected activities

In [27]:
selected_activities = ["Travel", "Restaurants", "Local Events"]

In [30]:
def create_subplot(activity_data, activity_name):
    p = figure(
        x_axis_type="datetime",
        title=f"Business Activity: {activity_name}",
        width=1500,
        height=600,
        toolbar_location="above",
    )
    p.add_layout(Legend(), "right")

    for idx, country in enumerate(selected_countries):
        df = activity_data[activity_data["gadm0_name"] == country]

        if not df.empty:
            p.line(
                x=df["ds"],
                y=df["activity_quantile"],
                line_width=2,
                color=color_palette[idx % len(color_palette)],  
                legend_label=country,
            )

    sub_title = figure(
    title="Source: Data for Good Meta",
    toolbar_location=None,
    width=800,
    height=40,
    )
    sub_title.title.align = "left"
    sub_title.title.text_font_size = "10pt"
    sub_title.title.text_font_style = "normal"
    sub_title.border_fill_alpha = 0
    sub_title.outline_line_width = 0

    p.legend.click_policy = "hide"
    return column(p, sub_title)


tabs = []
for activity in selected_activities:
    activity_data = filtered_data[filtered_data["business_vertical"] == activity]
    plot = create_subplot(activity_data, activity)
    tabs.append(TabPanel(child=plot, title=activity))

tabs_layout = Tabs(tabs=tabs, sizing_mode="scale_both")
show(tabs_layout)

Observation: Local events saw a decline in activity, while restaurants experienced an increase, which makes sense in the context of a pandemic.